# Load Shedding's Economic Ripple Effect
**Does the severity and duration of load shedding measurably move South African economic activity — and by how much, with what time lag?**

Portfolio project · BSc IT (Data Science) · July 2026

This notebook is the full analysis pipeline: data loading → cleaning → EDA → statistical testing → modelling → findings. Narrative is written between cells, research-report style.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats
import statsmodels.api as sm

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
RAW = "../data/raw/"
PROCESSED = "../data/processed/"
CHARTS = "../charts/"

## 2. Data loading

**Sources** (see `data/raw/SOURCES.md` for access dates):
- **Eskom Data Portal** — historical Manual Load Reduction (load shedding), official and authoritative
- **Stats SA P6242.1** — retail trade sales at constant 2019 prices, monthly

> Assumption log: we start with retail trade sales as the single economic indicator (brief §4.2) and monthly granularity, since retail data is monthly.

In [ ]:
# --- Eskom load shedding data ---
# TODO: update filename once the Eskom Data Portal export lands in data/raw/
ls_raw = pd.read_csv(RAW + "eskom_loadshedding.csv")
ls_raw.head()

In [ ]:
# --- Stats SA retail trade sales (P6242.1) ---
# TODO: update filename/sheet once downloaded from statssa.gov.za/?page_id=1847
retail_raw = pd.read_excel(RAW + "statssa_retail_trade_sales.xlsx")
retail_raw.head()

## 3. Cleaning & alignment

Goal: one tidy monthly dataframe with columns like
`month | ls_intensity (e.g. stage-hours or GWh shed) | retail_sales_const2019 | retail_yoy_pct`

In [ ]:
from data_prep import build_monthly_loadshedding, build_monthly_retail  # src/data_prep.py
import sys; sys.path.append("../src")
# monthly_ls = build_monthly_loadshedding(ls_raw)
# monthly_retail = build_monthly_retail(retail_raw)
# df = monthly_ls.merge(monthly_retail, on="month", how="inner")
# df.to_csv(PROCESSED + "monthly_merged.csv", index=False)
# df.head()

### Sanity check — do the shapes plausibly relate?
Plot both raw series on one chart *before* deeper analysis (brief §5, Phase 2).

In [ ]:
# fig, ax1 = plt.subplots()
# ax1.plot(df["month"], df["ls_intensity"], color="tab:red", label="Load shedding intensity")
# ax2 = ax1.twinx()
# ax2.plot(df["month"], df["retail_sales"], color="tab:blue", label="Retail sales (const 2019 prices)")
# plt.title("Load shedding vs retail trade sales")
# plt.savefig(CHARTS + "01_timeseries_overlay.png", dpi=150, bbox_inches="tight")

## 4. Exploratory & statistical analysis

Required (brief §3): **Pearson r, p-value, and 95% confidence interval** — correlation alone is not acceptable.

In [ ]:
def pearson_with_ci(x, y, alpha=0.05):
    """Pearson r with p-value and Fisher-z confidence interval."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    mask = ~(np.isnan(x) | np.isnan(y))
    x, y = x[mask], y[mask]
    r, p = stats.pearsonr(x, y)
    n = len(x)
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    zcrit = stats.norm.ppf(1 - alpha / 2)
    lo, hi = np.tanh(z - zcrit * se), np.tanh(z + zcrit * se)
    return {"r": r, "p": p, "n": n, "ci95": (lo, hi)}

# pearson_with_ci(df["ls_intensity"], df["retail_yoy_pct"])

### Lag analysis
Does a bad load shedding month predict a retail dip 1–4 months later? Shift the economic series backwards and re-test at each lag.

In [ ]:
# results = []
# for lag in range(0, 5):
#     res = pearson_with_ci(df["ls_intensity"][:len(df)-lag if lag else None],
#                           df["retail_yoy_pct"].shift(-lag).dropna())
#     results.append({"lag_months": lag, **res})
# lag_df = pd.DataFrame(results)
# lag_df

## 5. Modelling

### 5.1 OLS regression — quantify the impact
Target claim shape: *"each additional unit of load shedding intensity per month is associated with an X% change in retail sales, controlling for trend/seasonality."*

In [ ]:
# X = sm.add_constant(df[["ls_intensity"]])
# model = sm.OLS(df["retail_yoy_pct"], X, missing="drop").fit()
# print(model.summary())

### 5.2 Prophet forecast (recommended stretch — ONE model only, per brief §7)
Retail sales forecast with load shedding intensity as an extra regressor.

In [ ]:
# from prophet import Prophet
# pdf = df.rename(columns={"month": "ds", "retail_sales": "y"})
# m = Prophet()
# m.add_regressor("ls_intensity")
# m.fit(pdf)

## 6. Findings

_[Written after analysis: headline result, effect size, lag structure, significance.]_

## 7. Limitations & honest interpretation

- **Correlation ≠ causation** — load shedding intensity co-moves with the broader business cycle
- **Confounders**: COVID-19 lockdowns (2020–21), interest-rate cycle, fuel prices, riots (July 2021)
- **Regime note**: sustained load shedding largely ended in 2024 — the informative variance sits in 2019–2023
- Sample size at monthly granularity is modest; wide CIs are expected and should be reported honestly

## 8. Conclusion

_[TBD]_